# 5 · Training an agent

A market that reacts to what your agent does.

Historical data ignores you: an agent can buy a million shares of a name
that trades ten thousand a day and the tape carries on unchanged, so
whatever it learns about sizing is fiction. Here the book pushes back during
training.

**Prerequisites:** `pip install "pretium[rl]"`, which is gymnasium and numpy.

In [1]:
import numpy as np
import pretium as pt
from pretium.gym import TradingEnv

universe = pt.Universe.random(8, seed=7)
env = TradingEnv(universe=universe, seed=42, days=5)

print("observation space:", env.observation_space)
print("action space     :", env.action_space)

observation space: Box(-inf, inf, (17,), float64)
action space     : Box(-1.0, 1.0, (8,), float64)


## Actions are target weights

The action space is `Box(-1, 1)` per instrument: a target *portfolio weight*.
Negative is short.

This helps learnability. With share counts, a policy would have to learn
each instrument's price range first: a name that trades in single digits and
one that trades in the hundreds need very different action magnitudes for the
same economic position. Weights are dimensionless.

The environment converts a weight into the order that reaches it, and the
book decides the cost.

In [2]:
obs, info = env.reset(seed=42)
# reset's info carries the run's identity; the accounting fields appear
# once there has been a step to account for.
print("observation:", obs.shape, obs.dtype)
print("reset info :", info)

observation: (17,) float64
reset info : {'seed': 42, 'model_fingerprint': 'pt-v12'}


## One episode, random actions

Reward is the change in net worth, measured after the market moves, so it
already includes the cost of the agent's own footprint. Trading too large is
penalised by the book rather than by a tuned penalty term.

In [3]:
obs, info = env.reset(seed=42)
rng = np.random.default_rng(0)

total = 0.0
rows = []
while True:
    action = rng.uniform(-0.3, 0.3, size=env.action_space.shape)
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    rows.append((info["step"], reward, info["net_worth"], info["leverage"]))
    if terminated or truncated:
        break

print(f"{'step':>5s} {'reward':>12s} {'net worth':>14s} {'leverage':>9s}")
for step, r, nw, lev in rows[:8]:
    print(f"{step:5d} {r:12,.0f} {nw:14,.0f} {lev:9.3f}")
print(f"...\n{len(rows)} steps, total reward {total:,.0f}")

 step       reward      net worth  leverage
    1       -3,462        996,538     1.428
    2       -1,387        995,151     1.603
    3       -2,576        992,576     1.111
    4       -4,780        987,795     1.109
    5       -1,744        986,051     0.977
    6       -4,529        981,522     0.844
    7        2,722        984,244     1.507
    8       -6,584        977,660     1.186
...
30 steps, total reward -48,271


## Independent episodes

Each seed is a different market drawn from the same process, so an agent can
train on thousands of independent episodes rather than one historical path.

Note what varies: the universe seed holds the companies constant while the
simulation seed changes the market. That lets you ask whether a policy
generalises across markets rather than across companies.

In [4]:
CASH = 1_000_000.0          # TradingEnv's default starting cash

def episode_return(seed, policy_scale=0.3):
    e = TradingEnv(universe=universe, seed=seed, days=5, cash=CASH)
    obs, info = e.reset(seed=seed)
    start = CASH               # no positions yet, so net worth is the cash
    rng = np.random.default_rng(seed)
    while True:
        obs, r, term, trunc, info = e.step(
            rng.uniform(-policy_scale, policy_scale, size=e.action_space.shape))
        if term or trunc:
            break
    return (info["net_worth"] / start - 1) * 100

returns = [episode_return(s) for s in range(1, 11)]
print("ten independent markets, same random policy:")
print("  " + "  ".join(f"{r:+.2f}%" for r in returns))
print(f"\n  mean {np.mean(returns):+.2f}%   sd {np.std(returns):.2f}%")
print("\nThe spread IS the point: one episode measures the episode.")

ten independent markets, same random policy:
  -1.03%  -0.70%  -1.49%  -0.52%  +1.29%  +0.32%  +2.33%  +1.22%  +3.18%  +2.59%

  mean +0.72%   sd 1.56%

The spread IS the point: one episode measures the episode.


## Does size cost money?

Same market, same seed, same direction. Only the size changes.

In [5]:
def cost_of_size(scale, seed=42):
    e = TradingEnv(universe=universe, seed=seed, days=3, cash=CASH)
    obs, info = e.reset(seed=seed)
    start = CASH
    action = np.full(e.action_space.shape, scale)
    while True:
        obs, r, term, trunc, info = e.step(action)
        if term or trunc:
            break
    return (info["net_worth"] / start - 1) * 100, info["leverage"]

print(f"{'target weight':>14s} {'return':>9s} {'leverage':>9s}")
for scale in (0.02, 0.05, 0.10, 0.20, 0.40):
    ret, lev = cost_of_size(scale)
    print(f"{scale:14.2f} {ret:8.2f}% {lev:9.3f}")

 target weight    return  leverage
          0.02     0.21%     0.161
          0.05     0.51%     0.401
          0.10     1.05%     0.801
          0.20     2.32%     1.596


          0.40     1.28%     1.994


If size were free the last row would be the best one. An agent trained here
meets the constraint during training.

## Reproducibility

`reset(seed=n)` fixes the episode, so a training curve is something someone
else can re-run.

In [6]:
a = episode_return(99)
b = episode_return(99)
print(f"seed 99 twice: {a:+.6f}%  {b:+.6f}%   ->",
      "identical" if a == b else "DIFFERENT")

seed 99 twice: +0.374764%  +0.374764%   -> identical


## Caveats

**Good results here do not predict real returns.** The price process is a
known model, and a policy that discovers its structure will not transfer.
What does transfer is sizing discipline, respecting depth, and behaviour
under stress.

**Check the envelope before claiming a horizon.** Certification is 252 days,
and `pt.envelope.check(horizon_days=...)` refuses beyond it.

Next: **[6 · Execution and impact](06-execution-and-impact.ipynb)**.